## Setup
Import the libraries used for clustering and resolve the project's data/visuals directories relative to this notebook. This lets the notebook run locally, in JupyterLab, or in Colab (after cloning the repo) without any Google Drive mount.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Resolve paths relative to the repo root so this works regardless of whether
# the notebook is launched from the repo root or from the notebooks/ folder.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_DIR = PROJECT_ROOT / 'data'
VISUALS_DIR = PROJECT_ROOT / 'visuals'
VISUALS_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Data dir:', DATA_DIR)
print('Visuals dir:', VISUALS_DIR)

Loading Cleaned Dataset for Clustering

In [ ]:
df = pd.read_csv(DATA_DIR / 'airbnb_cleaned.csv')
print(df.shape)
df.head()

Select & Prepare Features for Clustering

In [ ]:
features = ['log_price', 'minimum_nights', 'number_of_reviews',
            'reviews_per_month', 'availability_365',
            'calculated_host_listings_count', 'neighbourhood_group', 'room_type']

df_cluster = df[features].copy()

le = LabelEncoder()
df_cluster['neighbourhood_group'] = le.fit_transform(df_cluster['neighbourhood_group'])
df_cluster['room_type'] = le.fit_transform(df_cluster['room_type'])

print(df_cluster.shape)
df_cluster.head()

Features Scaling

In [ ]:
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_cluster)
print("Scaling done. Shape:", df_scaled.shape)

Finding Optimal K (Elbow Method)

In [ ]:
inertia = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker='o', color='steelblue', linewidth=2)
plt.title('Elbow Method - Optimal K', fontsize=14)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.xticks(k_range)
plt.grid(True)
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'elbow_plot.png', dpi=150)
plt.show()

Silhouette Score

In [ ]:
sil_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(df_scaled)
    score = silhouette_score(df_scaled, labels)
    sil_scores.append(score)
    print(f"K={k} → Silhouette Score: {score:.4f}")

plt.figure(figsize=(8, 5))
plt.plot(k_range, sil_scores, marker='s', color='coral', linewidth=2)
plt.title('Silhouette Scores by K', fontsize=14)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.xticks(k_range)
plt.grid(True)
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'silhouette_plot.png', dpi=150)
plt.show()

Final K-Means with Best K

In [ ]:
best_k = 4  # Change this after seeing your plots

kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['cluster'] = kmeans_final.fit_predict(df_scaled)

print("Cluster distribution:")
print(df['cluster'].value_counts())

Profile the Clusters

In [ ]:
# Numeric summary per cluster
profile = df.groupby('cluster')[['price', 'minimum_nights', 'number_of_reviews',
                                   'reviews_per_month', 'availability_365']].mean().round(2)
print(profile)

# Room type distribution per cluster
print(pd.crosstab(df['cluster'], df['room_type'], normalize='index').round(2))

# Neighbourhood distribution per cluster
print(pd.crosstab(df['cluster'], df['neighbourhood_group'], normalize='index').round(2))

Visualizations
Bar chart — Average Price per Cluster:

In [ ]:
plt.figure(figsize=(7, 4))
df.groupby('cluster')['price'].mean().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Average Price per Cluster')
plt.xlabel('Cluster')
plt.ylabel('Average Price ($)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'price_per_cluster.png', dpi=150)
plt.show()

Room type distribution per cluster:

In [ ]:
room_ct = pd.crosstab(df['cluster'], df['room_type'])
room_ct.plot(kind='bar', figsize=(8, 5), edgecolor='black')
plt.title('Room Type Distribution per Cluster')
plt.xlabel('Cluster')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Room Type')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'roomtype_per_cluster.png', dpi=150)
plt.show()

Neighbourhood distribution per cluster:

In [ ]:
neigh_ct = pd.crosstab(df['cluster'], df['neighbourhood_group'])
neigh_ct.plot(kind='bar', figsize=(8, 5), edgecolor='black')
plt.title('Neighbourhood Group per Cluster')
plt.xlabel('Cluster')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Borough')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'neighbourhood_per_cluster.png', dpi=150)
plt.show()

Final Output

In [ ]:
df.to_csv(DATA_DIR / 'airbnb_with_clusters.csv', index=False)
print(f"Done! File saved to {DATA_DIR / 'airbnb_with_clusters.csv'}")